# 2.3 Multiple and Chained Assignment

## Overview

Python's compact assignment syntax hides several distinct semantic mechanisms: independent assignments, simultaneous-looking target-list assignment, chained assignment to several targets, and recursive unpacking. Although these forms are concise, they differ in right-hand-side evaluation, object sharing, target execution order, exception behavior, and their consequences for stateful numerical software.

In quantitative research, these differences become material when variables reference mutable portfolio positions, reusable configuration objects, market data, and financial state. This notebook develops a precise model of these operations and then uses that model to build and test a **multi-asset portfolio rebalancing engine**.

## Learning Objectives

1. Classify independent, target-list, chained, and nested assignments precisely.
2. Explain why the right-hand side is evaluated before assignment targets and why chained assignment evaluates it only once.
3. Apply pairwise swapping, tuple rotation, and multiple-name binding correctly.
4. Diagnose aliasing introduced by chained assignments to mutable objects.
5. Predict overlapping-target behavior using left-to-right target assignment.
6. Trace side effects from right-hand-side expressions, subscription targets, and attribute setters.
7. Distinguish unpacking failures from partial mutations after earlier targets were assigned.
8. Inspect assignment syntax using `ast` and observe implementation-specific CPython bytecode using `dis`.
9. Separate concise assignment syntax from application-level atomicity and transaction guarantees.
10. Design deterministic, invariant-checked portfolio allocations without accidentally sharing mutable state.
11. Produce automated unit tests and financial accounting verification.
12. Evaluate readability, correctness, and performance trade-offs in advanced assignment patterns.

## Scope and Boundaries

This section concentrates on **multiple and chained assignment**. Comprehensive sequence unpacking and starred-target grammar belong to **2.4**, augmented and in-place assignment to **2.5**, and assignment expressions (`:=`) to **2.7**. Those mechanisms appear here only where necessary to explain behavior and common pitfalls.

### Official References

- [Python Language Reference — Assignment statements](https://docs.python.org/3/reference/simple_stmts.html#assignment-statements)
- [Python Language Reference — Evaluation order](https://docs.python.org/3/reference/expressions.html#evaluation-order)
- [Python `ast` documentation](https://docs.python.org/3/library/ast.html)
- [Python `dis` documentation](https://docs.python.org/3/library/dis.html)


## 2.3.1 Assignment Forms and Formal Semantics

### Theoretical Foundations

The phrase *multiple assignment* can describe several different forms. Distinguish their structure rather than treating them as synonymous.

| Form | Example | Interpretation |
|---|---|---|
| Independent statements | `a = 1` then `b = 2` | Two statements, separate RHS evaluations |
| Target-list assignment | `a, b = 1, 2` | Evaluate RHS expressions; unpack into targets |
| Chained assignment | `a = b = expression` | Evaluate RHS once; assign same object to each target, left to right |
| Swapping / parallel-style assignment | `a, b = b, a` | Capture RHS references before assigning targets |
| Nested destructuring | `a, (b, c) = source` | Recursively distribute values to structured targets |
| Starred target | `first, *rest = source` | Variable-length unpacking; fully covered in 2.4 |

### Statement Model

For ordinary assignment with several target lists:

1. Evaluate the **entire right-hand-side expression list**.
2. Produce its resulting object (which may be a tuple when expressions are comma-separated).
3. Assign that object to **each target list from left to right**.
4. If a target list itself contains several targets, recursively distribute items according to its structure.

The right-hand side is evaluated **once for a chained assignment** such as `a = b = make_object()`; each target receives the **same object**, not an automatically constructed copy.

**Important:** The appearance of simultaneity in `a, b = b, a` arises from evaluating RHS values before applying the targets. It does **not** imply that arbitrary target assignments are atomic or protected against exceptions.


In [1]:
import ast

assignment_forms = {
    "independent": "a = 1\nb = 2",
    "target_list": "a, b = 1, 2",
    "chained": "a = b = create_payload()",
    "swap": "a, b = b, a",
    "nested": "symbol, (shares, price) = record",
    "starred": "first, *middle, last = prices",
}

for label, source in assignment_forms.items():
    parsed = ast.parse(source)
    kinds = [type(node).__name__ for node in parsed.body]
    target_counts = [len(node.targets) for node in parsed.body if isinstance(node, ast.Assign)]
    print(f"{label:<14} statements={len(parsed.body)}, nodes={kinds}, target-lists={target_counts}")

assert len(ast.parse(assignment_forms["independent"]).body) == 2
assert len(ast.parse(assignment_forms["chained"]).body[0].targets) == 2
assert len(ast.parse(assignment_forms["swap"]).body[0].targets) == 1


independent    statements=2, nodes=['Assign', 'Assign'], target-lists=[1, 1]
target_list    statements=1, nodes=['Assign'], target-lists=[1]
chained        statements=1, nodes=['Assign'], target-lists=[2]
swap           statements=1, nodes=['Assign'], target-lists=[1]
nested         statements=1, nodes=['Assign'], target-lists=[1]
starred        statements=1, nodes=['Assign'], target-lists=[1]


### **Analytical Interpretation**

**1. Source Structure.** Two independent statements become two AST statements. A chained assignment becomes one `Assign` node with multiple top-level `targets`.

**2. Target Lists vs. Number of Names.** `a, b = b, a` contains two names but **one top-level tuple target**. The distinction matters when analyzing source code.

**3. Chained Assignment.** `a = b = expression` contains two top-level assignment targets. The single RHS is evaluated once and its object is passed to both targets.

**4. Scope of This Chapter.** Nested and starred targets are introduced only as context for multiple assignment; exhaustive unpacking semantics appear in Section 2.4.

**5. Quantitative Engineering.** When evaluating financial calculations, do not infer whether data are independent from the number of variable names on the left-hand side. Inspect whether those names reference the same underlying mutable object.


## 2.3.2 Chained Assignment: Single RHS Evaluation and Left-to-Right Targets

### Theoretical Foundations

A chain such as `first = second = expression` is **one assignment statement**. Python evaluates `expression` once, then assigns the resulting object to `first`, followed by `second`.

- **Repeated evaluation does not occur:** `build()` is not called once per target.
- **Object sharing occurs:** each target receives the same returned object.
- **Target-specific operations remain observable:** a target may invoke an attribute setter or item assignment.
- **No automatic copying:** mutating a shared list or dictionary through one alias affects the other aliases.
- **No all-or-nothing guarantee:** if a later target rejects a value, an earlier assignment can remain in place.

Use chained assignment only when **intentional sharing** or assignment of a conceptually immutable value is clear and readable. Avoid it as a shortcut for constructing independent mutable containers.


In [2]:
events: list[str] = []

class TraceTarget:
    def __init__(self, name: str) -> None:
        object.__setattr__(self, "name", name)
        object.__setattr__(self, "payload", None)

    def __setattr__(self, name: str, value: object) -> None:
        events.append(f"assign:{self.name}.{name}")
        object.__setattr__(self, name, value)

def make_payload() -> list[int]:
    events.append("evaluate:RHS")
    return [10, 20]

left = TraceTarget("left")
right = TraceTarget("right")
left.payload = right.payload = make_payload()

assert events == ["evaluate:RHS", "assign:left.payload", "assign:right.payload"]
assert left.payload is right.payload
right.payload.append(30)
assert left.payload == [10, 20, 30]
print("Execution trace:", events)
print("Shared payload:", left.payload)


Execution trace: ['evaluate:RHS', 'assign:left.payload', 'assign:right.payload']
Shared payload: [10, 20, 30]


### **Analytical Interpretation**

**1. RHS Once.** The event `evaluate:RHS` appears once, demonstrating one evaluation for the entire chain.

**2. Left-to-Right Target Assignment.** `left.payload` is assigned before `right.payload`. Python's grammar and language reference define this target order.

**3. Custom Setters.** Each attribute target may execute user-defined assignment behavior. The language guarantees the ordering of the targets, not that setters are pure functions.

**4. Object Identity.** Both `payload` attributes identify exactly the same list object. Modifying the list through `right` also changes what `left` observes.

**5. Quant Application.** `live_positions = backtest_positions = {}` would share one dictionary. That may contaminate an experiment unless intentional. The correct implementation depends on an explicit data-ownership policy.


## 2.3.3 Parallel-Style Assignment, Swapping, and Rotation

### Theoretical Foundations

An assignment such as `a, b = b, a` is often called *simultaneous assignment* or *parallel assignment*, but its Python implementation follows the ordinary assignment rules:

1. Evaluate the RHS expression list in order.
2. Produce the RHS values as a combined result.
3. Assign to LHS targets in order.

This makes swaps safe without a temporary name. It also enables rotations and multi-component restructuring:

- Pair swap: `a, b = b, a`
- Three-way rotation: `a, b, c = b, c, a`
- Financial weight exchange: `weights["AAPL"], weights["MSFT"] = weights["MSFT"], weights["AAPL"]`

**Subtlety:** The tuple-like snapshot of RHS values is not necessarily a deep copy. If those values are references to mutable objects, the resulting bindings can still alias those objects.

**Readability requirement:** A multi-target assignment should express a single coherent transformation; avoid using it to bundle unrelated state changes solely for brevity.


In [3]:
# Independent values can be swapped without an explicit temporary.
a, b = "AAPL", "MSFT"
a, b = b, a
assert (a, b) == ("MSFT", "AAPL")

# Rotation takes RHS values before any target is rebound.
first, second, third = 10, 20, 30
first, second, third = second, third, first
assert (first, second, third) == (20, 30, 10)

# A meaningful financial application: exchange two target allocations.
target_weights = {"AAPL": 0.45, "MSFT": 0.35, "JPM": 0.20}
total_before = sum(target_weights.values())
target_weights["AAPL"], target_weights["MSFT"] = (
    target_weights["MSFT"], target_weights["AAPL"]
)
assert target_weights == {"AAPL": 0.35, "MSFT": 0.45, "JPM": 0.20}
assert sum(target_weights.values()) == total_before

# The values on the RHS may themselves be mutable references.
left_book, right_book = ["old-left"], ["old-right"]
left_book, right_book = right_book, left_book
assert left_book == ["old-right"] and right_book == ["old-left"]
print("Rotated values:", (first, second, third))
print("Reallocated weights:", target_weights)


Rotated values: (20, 30, 10)
Reallocated weights: {'AAPL': 0.35, 'MSFT': 0.45, 'JPM': 0.2}


### **Analytical Interpretation**

**1. Swap Correctness.** Python evaluates the previous values of both names before rebinding either target. That prevents one target update from destroying the value needed by the other.

**2. Rotation.** The three-way rotation is a generalization of the same right-hand-side capture mechanism.

**3. Financial Invariant.** Exchanging two target weights changes allocation membership but preserves the sum of all portfolio weights.

**4. Mutable Objects.** Swapping two lists exchanges references; it does not create copies of the list contents.

**5. Implementation Discipline.** Multi-target assignment offers concise notation for swaps. It should not be used to imply transactional guarantees for cash, positions, and execution logs.


## 2.3.4 Mutable Aliasing and Independent Object Construction

### Theoretical Foundations

Chained assignment is particularly hazardous when the RHS creates a mutable object:

```python
research_book = execution_book = {}
```

Both names refer to the same dictionary. Similarly, list repetition such as `[[]] * 3` produces multiple references to one inner list; it does not construct three independent inner lists.

### Important Cases

- `a = b = []`: one newly created list, two names.
- `a, b = [], []`: two distinct list expressions, two newly created lists.
- `records = [[]] * 3`: three references to one inner list.
- `records = [[] for _ in range(3)]`: three independent inner lists.
- `a = b = 100`: both names refer to the RHS value, but do not use small-integer interning or identity reuse to infer a general allocation policy.

Aliasing and equality are different. Immutable values can be shared safely with respect to mutation, but do not infer general identity relationships solely from equal immutable values.


In [4]:
# Same object bound to two names.
research_book = execution_book = {"AAPL": 100}
assert research_book is execution_book
execution_book["AAPL"] = 120
assert research_book["AAPL"] == 120

# Independent objects from separate RHS expressions.
model_book, audit_book = {"AAPL": 100}, {"AAPL": 100}
assert model_book == audit_book
assert model_book is not audit_book
model_book["AAPL"] = 115
assert audit_book["AAPL"] == 100

# The repeated reference pitfall.
shared_scenarios = [[]] * 3
shared_scenarios[0].append("shock")
assert shared_scenarios == [["shock"], ["shock"], ["shock"]]

independent_scenarios = [[] for _ in range(3)]
independent_scenarios[0].append("shock")
assert independent_scenarios == [["shock"], [], []]
print("Shared scenarios:", shared_scenarios)
print("Independent scenarios:", independent_scenarios)


Shared scenarios: [['shock'], ['shock'], ['shock']]
Independent scenarios: [['shock'], [], []]


### **Analytical Interpretation**

**1. Alias Creation.** The chained assignment shares one dictionary between `research_book` and `execution_book`.

**2. Independent Construction.** Two dictionary expressions in a RHS tuple produce two independent dictionaries, preserving the audit copy after the model copy is updated.

**3. Container Multiplication.** `* 3` repeats references to the inner list; a nested mutable object is not cloned.

**4. Comprehension Construction.** The list comprehension runs the inner list constructor on each iteration, producing separate inner containers.

**5. Quant Consequence.** Repeated-reference structures can cause all scenario portfolios to change when only one was intended to change. This can invalidate simulation independence.

**6. Best Practice.** For independently mutable research objects, create each one explicitly, use a comprehension, or apply an appropriate copy strategy at the data-ownership boundary.


## 2.3.5 Overlapping Targets and Partial Assignment Failure

### Theoretical Foundations

A multi-target assignment applies targets **from left to right**. Earlier target assignments can affect the interpretation of later target expressions.

Classic overlapping-target examples:

```python
i, values[i] = 1, 99
values[i], i = 99, 1
```

They use the same RHS values, but update different list elements because `i` is rebound at different points.

### Failure Semantics

A chained assignment is **not an atomic transaction**. If target 1 succeeds and target 2 raises an exception, target 1 can remain modified. The RHS may also have performed side effects before the first target assignment.

In contrast, an unpacking failure that occurs during extraction of the RHS elements may happen *before* any targets are assigned, although nested patterns can still produce partially completed outer assignments.

**Engineering implication:** In a financial ledger, never use a comma-separated target list as a substitute for validated state transitions and rollback discipline.


In [5]:
# Overlapping targets: the earlier binding changes the later index.
values = [0, 1]
i = 0
i, values[i] = 1, 99
assert (i, values) == (1, [0, 99])

values = [0, 1]
i = 0
values[i], i = 99, 1
assert (i, values) == (1, [99, 1])

# A later failing target does not reverse the earlier target.
class RejectingTarget:
    @property
    def prohibited(self) -> int:
        return 0

    @prohibited.setter
    def prohibited(self, value: object) -> None:
        raise ValueError("Denied by target setter")

holder = RejectingTarget()
successful: object = "previous"
try:
    successful = holder.prohibited = {"order": "T001"}
except ValueError:
    pass
assert successful == {"order": "T001"}

print("First overlapping order:", [0, 99])
print("Second overlapping order:", [99, 1])
print("Earlier target survives later failure:", successful)


First overlapping order: [0, 99]
Second overlapping order: [99, 1]
Earlier target survives later failure: {'order': 'T001'}


### **Analytical Interpretation**

**1. Target Ordering.** `i, values[i] = 1, 99` first rebinds `i` to 1; only then does it evaluate the later subscript target. The list element at index 1 changes.

**2. Order Reversal.** `values[i], i = 99, 1` updates the list at the old index 0 before rebinding `i`.

**3. Chained Target Failure.** Assignment to `successful` completes before the property setter rejects the value. Catching the exception does not restore the earlier binding.

**4. Financial Implication.** Cash and positions should not be treated as a single transaction solely because their updates appear in one assignment statement.

**5. Separation of Guarantees.** Expression evaluation order is a language guarantee. Atomic commits, rollback, concurrency, and durability are **application-level** guarantees requiring separate design.


## 2.3.6 Evaluation Order and Side-Effectful Targets

### Theoretical Foundations

Assignment expressions can have observable computation steps:

- RHS functions are evaluated in the order they appear.
- An RHS expression can alter application state before any target is changed.
- LHS attribute or subscription expressions can invoke functions and custom methods.
- Target lists are processed from left to right.
- A target's setter can reject or transform an assigned value.

When reasoning about a statement such as:

```python
get_container()[get_key()], state.version = make_value(), get_version()
```

the two RHS calls execute first; the left target is then evaluated and assigned, followed by the right target.

**Method:** build deterministic traces and assertions rather than relying on intuition or CPython bytecode as a cross-version specification.


In [6]:
trace: list[str] = []

class AuditedMapping:
    def __init__(self) -> None:
        self.data: dict[str, int] = {}

    def __setitem__(self, key: str, value: int) -> None:
        trace.append(f"STORE_ITEM:{key}={value}")
        self.data[key] = value

class AuditedVersion:
    def __init__(self) -> None:
        object.__setattr__(self, "version", 0)

    def __setattr__(self, name: str, value: int) -> None:
        trace.append(f"STORE_ATTR:{name}={value}")
        object.__setattr__(self, name, value)

mapping = AuditedMapping()
version_state = AuditedVersion()

def log_value(label: str, value: int) -> int:
    trace.append(f"RHS:{label}")
    return value

def get_mapping() -> AuditedMapping:
    trace.append("TARGET:mapping")
    return mapping

def get_key() -> str:
    trace.append("TARGET:key")
    return "AAPL"

get_mapping()[get_key()], version_state.version = (
    log_value("quantity", 100),
    log_value("version", 1),
)

assert trace == [
    "RHS:quantity", "RHS:version", "TARGET:mapping", "TARGET:key",
    "STORE_ITEM:AAPL=100", "STORE_ATTR:version=1",
]
assert mapping.data["AAPL"] == 100
assert version_state.version == 1
print("\n".join(trace))


RHS:quantity
RHS:version
TARGET:mapping
TARGET:key
STORE_ITEM:AAPL=100
STORE_ATTR:version=1


### **Analytical Interpretation**

**1. RHS First.** Both RHS functions are called in their expression-list order before either target expression is evaluated.

**2. Target Evaluation.** The mapping expression is evaluated, then its key expression, then `__setitem__` performs the item assignment.

**3. Subsequent Target.** Only after item assignment does Python update the version attribute through `__setattr__`.

**4. Audit Fidelity.** The trace is deterministic for this example and directly checks the intended language semantics.

**5. Risks.** If setting the version raised an exception, the mapping would already contain the new position. This requires an explicit consistency boundary in production systems.


## 2.3.7 Unpacking Failure Boundaries (Preview of Section 2.4)

### Theoretical Foundations

Target-list assignment requires values to match the structure of the targets. When there are too few or too many items for an unstarred target list, Python raises `ValueError`.

**Two useful distinctions:**

1. **Top-level unpacking mismatch:** extraction of the expected items fails before corresponding name targets are bound.
2. **Nested target-list failure:** an earlier outer target may already have been bound before unpacking a later nested target fails.

The RHS iterable itself may already have been consumed, and a generator may have executed side effects. Therefore, *absence of completed target bindings* is not equivalent to *absence of side effects*.

A starred target changes the permitted number of input elements and is treated comprehensively in Section 2.4.


In [7]:
# Top-level mismatch: no name-target update when unpacking fails.
first, second = "old-first", "old-second"
try:
    first, second = [10]
except ValueError:
    pass
assert (first, second) == ("old-first", "old-second")

# Nested mismatch: first outer binding can be completed.
primary, detail_a, detail_b = "old-primary", "old-a", "old-b"
try:
    primary, (detail_a, detail_b) = ("AAPL", [50])
except ValueError:
    pass
assert primary == "AAPL"
assert (detail_a, detail_b) == ("old-a", "old-b")

# RHS iterator side effects can occur despite failed unpacking.
events = []
def source_iter():
    events.append("yield:first")
    yield 10
    events.append("finish")

left_name, right_name = "L", "R"
try:
    left_name, right_name = source_iter()
except ValueError:
    pass
assert (left_name, right_name) == ("L", "R")
assert events == ["yield:first", "finish"]
print("Nested result:", primary, detail_a, detail_b)
print("Iterator trace:", events)


Nested result: AAPL old-a old-b
Iterator trace: ['yield:first', 'finish']


### **Analytical Interpretation**

**1. Top-Level Mismatch.** The first unpacking operation fails before any corresponding names are changed.

**2. Nested Failure.** The outer target `primary` is bound to `"AAPL"` before unpacking the nested target fails. Earlier outer bindings are not rolled back.

**3. Iterator Consumption.** Unpacking an iterator consumes it, potentially invoking code and producing side effects even when the resulting name assignment fails.

**4. No Universal Atomicity.** The distinction between extraction and target assignment creates failure boundaries that must be understood explicitly.

**5. Portfolio Implication.** Do not unpack partly validated external trade records directly into authoritative portfolio state; validate shape and contents before committing changes.


## 2.3.8 AST and CPython Bytecode Inspection

### Theoretical Foundations

The `ast` module provides a structural representation of assignment forms. For ordinary assignment, `ast.Assign` stores:

- `targets`: a list of *top-level target lists*;
- `value`: the RHS expression.

A single tuple target may contain many `Name` nodes. A chained assignment contains multiple top-level targets, even if each one is a simple name.

Examples:

```python
a, b = b, a         # len(Assign.targets) == 1
x = y = create()    # len(Assign.targets) == 2
```

`dis` reveals implementation-specific CPython bytecode; it is useful for diagnostics, not a substitute for the Python Language Reference. Instruction sets differ across Python versions and implementations.


In [8]:
import ast
import dis

source = """
a, b = b, a
x = y = make_payload()
left, (middle, right) = [1, [2, 3]]
"""
parsed = ast.parse(source)
print("AST assignment structures")
for node in parsed.body:
    if isinstance(node, ast.Assign):
        print(
            "targets:", len(node.targets),
            "kinds:", [type(target).__name__ for target in node.targets],
            "RHS:", type(node.value).__name__,
        )

assert len(parsed.body[0].targets) == 1
assert isinstance(parsed.body[0].targets[0], ast.Tuple)
assert len(parsed.body[1].targets) == 2

def swap_two(a: int, b: int) -> tuple[int, int]:
    a, b = b, a
    return a, b

print("\nCPython disassembly (version-dependent):")
dis.dis(swap_two)
assert swap_two(3, 7) == (7, 3)


AST assignment structures
targets: 1 kinds: ['Tuple'] RHS: Tuple
targets: 2 kinds: ['Name', 'Name'] RHS: Call
targets: 1 kinds: ['Tuple'] RHS: List

CPython disassembly (version-dependent):
 23           RESUME                   0

 24           LOAD_FAST_LOAD_FAST     16 (b, a)
              STORE_FAST_STORE_FAST   16 (b, a)

 25           LOAD_FAST_LOAD_FAST      1 (a, b)
              BUILD_TUPLE              2
              RETURN_VALUE


### **Analytical Interpretation**

**1. AST Top-Level Targets.** A tuple destructuring statement is represented with one top-level `Tuple` target. A chained assignment contains several top-level target nodes.

**2. Compiler Structure.** The AST records the grammar of the program, not whether an object is mutable or whether a target setter will fail.

**3. Bytecode.** CPython may implement swapping or unpacking with different instruction sequences across releases. Those details are not stable public APIs.

**4. Verification.** Runtime assertions verify the observable values independently of disassembly output.

**5. Engineering Use.** AST-aware linters can flag chained assignments to mutable literals and complex overlapping targets, but static warnings must be interpreted in context.


## 2.3.9 Advanced Pitfalls, Correctness, and Code Review

### Theoretical Foundations

### Common Failure Modes

1. **Implicit aliasing:** `a = b = {}` shares a mutable dictionary.
2. **Misleading simultaneity:** target-list syntax does not create a transaction.
3. **Overlapping targets:** an earlier assignment changes a later index or target expression.
4. **Nested unpacking failures:** earlier outer assignments can remain after a later failure.
5. **Iterator side effects:** failed unpacking may have consumed input.
6. **Using identity for numbers:** value equality is not a reliable reason to expect or compare immutable object identities.
7. **Duplicated updates:** combining unrelated state changes into one statement hides dependencies and errors.
8. **Unreviewed input:** source records may be mutable or malformed; bind validated snapshots instead.
9. **Notebook hidden state:** assignment order matters across Jupyter cells; execute from a clean kernel.
10. **Library-specific semantics:** array and dataframe libraries can implement views, copies, and chained indexing differently; core Python assignment syntax alone does not establish these external behaviors.

### Review Checklist

- Is RHS evaluated exactly as intended?
- Does each target refer to the intended object or container?
- Are shared references intentional?
- Could an earlier target update alter a later target?
- Can a setter or indexing operation raise after earlier mutations?
- Are the financial units and invariants explicit?
- Does the design require staging, locking, persistence, or rollback?

**Rule:** Prefer separate named intermediate values when the sequence of financial state changes matters more than compact syntax.


In [9]:
from math import isclose

# Anti-pattern: two reports accidentally share one mutable metrics dictionary.
model_metrics = audit_metrics = {"gross_return": 0.05}
model_metrics["gross_return"] = 0.10
assert audit_metrics["gross_return"] == 0.10

# Explicit independent report objects prevent this aliasing.
model_metrics, audit_metrics = (
    {"gross_return": 0.05},
    {"gross_return": 0.05},
)
model_metrics["gross_return"] = 0.10
assert audit_metrics["gross_return"] == 0.05

# Report an actual invariant instead of assuming a swap is correct.
weights = {"AAPL": 0.55, "MSFT": 0.25, "JPM": 0.20}
initial_total = sum(weights.values())
weights["AAPL"], weights["MSFT"] = weights["MSFT"], weights["AAPL"]
assert isclose(sum(weights.values()), initial_total, abs_tol=1e-12)
assert isclose(initial_total, 1.0, abs_tol=1e-12)
print("Independent audit metrics:", audit_metrics)
print("Weights after allocation swap:", weights)


Independent audit metrics: {'gross_return': 0.05}
Weights after allocation swap: {'AAPL': 0.25, 'MSFT': 0.55, 'JPM': 0.2}


### **Analytical Interpretation**

**1. The Alias Trap.** Chaining a mutable dictionary makes the audit record sensitive to subsequent model updates.

**2. Independent Construction.** Separate dictionary expressions produce independently mutable records and protect the reference metrics from unintended changes.

**3. Numerical Invariants.** Swapping financial allocations preserves their total; this property is checked explicitly with a floating-point tolerance.

**4. Quantitative Integrity.** A correctly formatted line of assignment code can still invalidate historical comparisons when it introduces unintended shared state.

**5. Design Recommendation.** Use assignment shorthand for coherent local transformations, not for implicit multi-object coordination or undocumented sharing.


## 2.3.10 Practical Application — Multi-Asset Portfolio Rebalancing Engine

### Project Overview

**Project:** *Assignment-Safe Portfolio Rebalance and State Publication*

### Objective

Build an advanced, deterministic, standard-library portfolio rebalancer that demonstrates the correct use of multiple assignment while preserving coherent financial state. The key question is:

> How can several related financial values be computed through Python assignment without unintentionally sharing mutable objects or exposing a partially updated portfolio?

### Pipeline

```text
Validated holdings, cash, and target quantities
                    |
                    v
           Independent input copies
                    |
                    v
        Compute all old/new position pairs
                    |
                    v
        Derive ordered trade deltas and fees
                    |
                    v
     Apply sells before buys to staged cash
                    |
                    v
     Verify funding and equity conservation
                    |
              +-----+-----+
              |           |
            Fail        Pass
              |           |
            Reject      Publish a new
            without     immutable state
            mutation    under one lock
```


### Financial Definitions

For each asset $i$, assuming a single common mark/execution price $P_i$:

$$
\Delta q_i = q_i^{\mathrm{target}} - q_i^{\mathrm{old}}, \qquad
N_i = |\Delta q_i| P_i
$$

A proportional execution fee, measured in basis points, is:

$$
F_i = N_i \frac{\mathrm{FeeBps}}{10{,}000}
$$

A **sell** produces cash inflow $N_i - F_i$, while a **buy** produces cash outflow $N_i + F_i$.

With a fixed price vector, no financing costs, and no external contributions:

$$
E_{\mathrm{pre}} = E_{\mathrm{post}} + \sum_i F_i
$$

where portfolio equity is defined as:

$$
E = \mathrm{Cash} + \sum_i q_i P_i
$$

### Design Constraints

1. Long-only, fully paid holdings; no borrowing or short sales.
2. Positive finite execution prices; strictly positive or zero integer quantities.
3. Prices used for marks equal the example's execution prices.
4. No market impact, spreads, taxes, dividends, or financing.
5. Every asset in the existing or target portfolio must have a valid price.
6. Sell trades are staged before buy trades; this is a modeling assumption, not a live execution scheduler.
7. Each successful, nonempty rebalance publishes one new immutable state object.
8. Failed rebalances leave the existing state unchanged.
9. No-op requests do not increment the version.
10. The synchronization lock only protects cooperating operations within one process.

### Connection to Assignment Semantics

- `old_qty, new_qty = ...`: paired RHS evaluation before local name binding.
- `delta, notional = ...`: coherent multi-value derivation with explicit units.
- `candidate = PortfolioState(...)`: independent immutable candidate construction.
- `self._state = candidate`: a **single state-reference publication** under a lock, avoiding separate public cash and position attribute updates.

### What This Does Not Prove

This is not a real exchange simulator, database transaction manager, or crash-durable trading ledger. No concurrency guarantee applies to direct access to undocumented internal attributes. Floating-point bookkeeping is adequate for this educational example, not accounting settlement.


In [10]:
from collections.abc import Mapping
from dataclasses import dataclass
from math import fsum, isclose, isfinite
from threading import RLock

BPS_PER_UNIT = 10_000.0


def valid_real(value: object) -> bool:
    """Check for finite built-in real numeric values, excluding bool."""
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
        and isfinite(value)
    )


def normalize_quantities(values: Mapping[str, int]) -> dict[str, int]:
    """Return an independent, positive-quantity mapping."""
    if not isinstance(values, Mapping):
        raise TypeError("Positions must be a mapping")
    normalized: dict[str, int] = {}
    for ticker, shares in values.items():
        if not isinstance(ticker, str) or not ticker.strip():
            raise ValueError("Ticker must be a nonempty string")
        if isinstance(shares, bool) or not isinstance(shares, int) or shares < 0:
            raise ValueError(f"Invalid share quantity for {ticker}")
        if shares:
            normalized[ticker] = shares
    return normalized


@dataclass(frozen=True, slots=True)
class PortfolioState:
    """An immutable long-only portfolio snapshot."""
    version: int
    cash_usd: float
    positions: tuple[tuple[str, int], ...]


@dataclass(frozen=True, slots=True)
class TradeDelta:
    """A hypothetical rebalance instruction at a fixed mark price."""
    ticker: str
    old_shares: int
    target_shares: int
    delta_shares: int
    price_usd: float
    notional_usd: float
    fee_usd: float


@dataclass(frozen=True, slots=True)
class RebalanceReport:
    """A committed/no-op transition and financial reconciliation."""
    before: PortfolioState
    after: PortfolioState
    trades: tuple[TradeDelta, ...]
    total_fees_usd: float
    pre_equity_usd: float
    post_equity_usd: float
    committed: bool

print("Immutable rebalance data models created.")


Immutable rebalance data models created.


### **Analytical Interpretation**

**1. Typed State.** `PortfolioState` combines version, cash, and positions into one immutable data record.

**2. No Mutable Nesting.** Position holdings are stored as a tuple of `(ticker, shares)` tuples. The position snapshot does not retain a mutable caller-supplied dictionary.

**3. Input Normalization.** `normalize_quantities` rejects Booleans and negative quantities, removes zero quantities, and constructs a new mapping.

**4. Multiple Assignment Scope.** The subsequent engine will use multi-target assignment for *local calculations*, not for publishing several independently mutable public fields.

**5. Unit Discipline.** Trade notional, commission, cash, and equity explicitly use USD. Fees supplied in basis points are converted with the constant 10,000.


In [11]:
class RebalanceBook:
    """Publish validated portfolio rebalance states under one lock."""

    def __init__(self, cash_usd: float, positions: Mapping[str, int]) -> None:
        if not valid_real(cash_usd) or cash_usd < 0:
            raise ValueError("Initial cash must be finite and nonnegative")
        clean_positions = normalize_quantities(positions)
        self._lock = RLock()
        self._state = PortfolioState(
            version=0,
            cash_usd=float(cash_usd),
            positions=tuple(sorted(clean_positions.items())),
        )

    def snapshot(self) -> PortfolioState:
        """Return the immutable, current portfolio state."""
        with self._lock:
            return self._state

    def rebalance(
        self,
        targets: Mapping[str, int],
        prices_usd: Mapping[str, float],
        *,
        fee_bps: float = 10.0,
    ) -> RebalanceReport:
        """Stage, reconcile, and publish a long-only rebalance."""
        if not valid_real(fee_bps) or fee_bps < 0:
            raise ValueError("fee_bps must be finite and nonnegative")
        if not isinstance(prices_usd, Mapping):
            raise TypeError("prices_usd must be a mapping")

        # Capture independent input dictionaries so the engine does not
        # mutate any caller-owned mapping in its ordinary operation.
        target_copy = normalize_quantities(dict(targets))
        prices_copy = dict(prices_usd)

        with self._lock:
            before = self._state
            old_cash, old_positions = (
                before.cash_usd,
                dict(before.positions),
            )
            tickers = sorted(set(old_positions) | set(target_copy))
            prices: dict[str, float] = {}

            for ticker in tickers:
                if ticker not in prices_copy:
                    raise ValueError(f"Missing price for {ticker}")
                price = prices_copy[ticker]
                if not valid_real(price) or price <= 0:
                    raise ValueError(f"Invalid price for {ticker}")
                prices[ticker] = float(price)

            pre_equity = old_cash + fsum(
                shares * prices[ticker]
                for ticker, shares in old_positions.items()
            )
            if not isfinite(pre_equity):
                raise OverflowError("Nonfinite portfolio value")

            trades: list[TradeDelta] = []
            for ticker in tickers:
                # Multiple assignment captures both previous and desired
                # quantities before the new delta is computed.
                old_qty, new_qty = (
                    old_positions.get(ticker, 0),
                    target_copy.get(ticker, 0),
                )
                delta, notional = (
                    new_qty - old_qty,
                    abs(new_qty - old_qty) * prices[ticker],
                )
                if delta == 0:
                    continue
                fee = notional * float(fee_bps) / BPS_PER_UNIT
                if not all(isfinite(x) for x in (notional, fee)):
                    raise OverflowError("Nonfinite trade calculation")
                trades.append(
                    TradeDelta(
                        ticker=ticker,
                        old_shares=old_qty,
                        target_shares=new_qty,
                        delta_shares=delta,
                        price_usd=prices[ticker],
                        notional_usd=notional,
                        fee_usd=fee,
                    )
                )

            if not trades:
                return RebalanceReport(
                    before=before,
                    after=before,
                    trades=(),
                    total_fees_usd=0.0,
                    pre_equity_usd=pre_equity,
                    post_equity_usd=pre_equity,
                    committed=False,
                )

            # Sell first, then buy. All operations are local staging.
            staged_cash = old_cash
            for trade in trades:
                if trade.delta_shares < 0:
                    staged_cash += trade.notional_usd - trade.fee_usd
            for trade in trades:
                if trade.delta_shares > 0:
                    staged_cash -= trade.notional_usd + trade.fee_usd

            if not isfinite(staged_cash):
                raise OverflowError("Nonfinite cash balance")
            if staged_cash < 0:
                raise ValueError("Insufficient cash to fund rebalance")

            total_fees = fsum(trade.fee_usd for trade in trades)
            post_equity = staged_cash + fsum(
                quantity * prices[ticker]
                for ticker, quantity in target_copy.items()
            )
            if not isfinite(post_equity):
                raise OverflowError("Nonfinite post-trade equity")
            if not isclose(
                pre_equity,
                post_equity + total_fees,
                rel_tol=1e-12,
                abs_tol=1e-8,
            ):
                raise ArithmeticError("Equity reconciliation failed")

            candidate = PortfolioState(
                version=before.version + 1,
                cash_usd=staged_cash,
                positions=tuple(sorted(target_copy.items())),
            )
            # Publish one validated immutable state reference.
            self._state = candidate
            return RebalanceReport(
                before=before,
                after=candidate,
                trades=tuple(trades),
                total_fees_usd=total_fees,
                pre_equity_usd=pre_equity,
                post_equity_usd=post_equity,
                committed=True,
            )

print("Rebalance engine initialized.")


Rebalance engine initialized.


### **Analytical Interpretation**

**1. Old and Target Quantities.** The statement `old_qty, new_qty = ...` captures two numeric values for one ticker before any candidate state is published.

**2. Trade Derivation.** `delta, notional = ...` computes signed quantity change and unsigned trade notional with explicit USD units.

**3. No Shared Dictionary.** `target_copy` and `old_positions` are independent dictionaries. The new immutable state owns a tuple representation, not a caller's mutable mapping.

**4. Execution Order.** Sales are staged first, so sell proceeds may fund buys. This is a modeling choice and does not model price movement between fills.

**5. Financial Invariant.** Under fixed execution/mark prices, fees are the only modeled cause of equity reduction. `pre_equity ≈ post_equity + total_fees` is checked before commit.

**6. One State Reference.** The candidate is constructed and validated before the one authoritative assignment `self._state = candidate`.

**7. Locking.** The lock protects the snapshot and rebalance methods that cooperate with it. It does not provide process-crash durability, distributed consistency, or protection from unsupported direct internal mutation.

**8. Failure Preservation.** Validation and financial reconciliation failures occur before publication; the authoritative state remains unchanged under these intended failure paths.


In [12]:
book = RebalanceBook(
    cash_usd=2_000.0,
    positions={"AAPL": 100, "MSFT": 50},
)
prices = {"AAPL": 100.0, "MSFT": 200.0, "JPM": 150.0}
target = {"AAPL": 70, "MSFT": 65, "JPM": 10}

before = book.snapshot()
result = book.rebalance(target, prices, fee_bps=10.0)

after = book.snapshot()
assert result.committed is True
assert result.before == before
assert result.after == after
assert after.version == 1
assert dict(after.positions) == target
assert isclose(after.cash_usd, 492.5, abs_tol=1e-9)
assert isclose(result.total_fees_usd, 7.5, abs_tol=1e-9)
assert isclose(result.pre_equity_usd, 22_000.0, abs_tol=1e-9)
assert isclose(result.post_equity_usd, 21_992.5, abs_tol=1e-9)

print("MULTI-ASSET REBALANCE")
print("-" * 78)
for trade in result.trades:
    print(
        f"{trade.ticker:<7} old={trade.old_shares:>4} "
        f"new={trade.target_shares:>4} delta={trade.delta_shares:>4} "
        f"notional=${trade.notional_usd:>8,.2f} "
        f"fee=${trade.fee_usd:.2f}"
    )
print("-" * 78)
print(f"Pre-equity:  ${result.pre_equity_usd:,.2f}")
print(f"Trade fees:  ${result.total_fees_usd:,.2f}")
print(f"Post-equity: ${result.post_equity_usd:,.2f}")
print(f"Ending cash: ${after.cash_usd:,.2f}")

# A failed oversized rebalance does not change the published state.
previous = book.snapshot()
try:
    book.rebalance({"AAPL": 1_000, "MSFT": 65, "JPM": 10}, prices)
except ValueError as error:
    print("\nRejected oversized rebalance:", error)
assert book.snapshot() == previous

# No-op request preserves version and state.
noop = book.rebalance(target, prices)
assert noop.committed is False
assert noop.after is previous
print("No-op version:", book.snapshot().version)


MULTI-ASSET REBALANCE
------------------------------------------------------------------------------
AAPL    old= 100 new=  70 delta= -30 notional=$3,000.00 fee=$3.00
JPM     old=   0 new=  10 delta=  10 notional=$1,500.00 fee=$1.50
MSFT    old=  50 new=  65 delta=  15 notional=$3,000.00 fee=$3.00
------------------------------------------------------------------------------
Pre-equity:  $22,000.00
Trade fees:  $7.50
Post-equity: $21,992.50
Ending cash: $492.50

Rejected oversized rebalance: Insufficient cash to fund rebalance
No-op version: 1



### Expected Results

#### Trade-Level Plan

| Asset | Initial Shares | Target Shares | Delta | Trade Notional | Fee (10 bps) |
|---|---:|---:|---:|---:|---:|
| AAPL | 100 | 70 | -30 | &#36;3,000.00 | &#36;3.00 |
| JPM | 0 | 10 | +10 | &#36;1,500.00 | &#36;1.50 |
| MSFT | 50 | 65 | +15 | &#36;3,000.00 | &#36;3.00 |

#### Accounting Summary

| Metric | Expected Value |
|---|---:|
| Initial cash | &#36;2,000.00 |
| Initial marked positions | &#36;20,000.00 |
| Pre-trade equity | &#36;22,000.00 |
| Total trade notional | &#36;7,500.00 |
| Total transaction fees | &#36;7.50 |
| Final cash | &#36;492.50 |
| Final marked positions | &#36;21,500.00 |
| Post-trade equity | &#36;21,992.50 |
| Published version | 1 |
| Oversized rebalance | Rejected, prior state preserved |
| Repeat identical target | No-op, version unchanged |

#### Accounting Identity

$$
22{,}000.00 = 21{,}992.50 + 7.50
$$

**Interpretation:** The exercise is a deterministic allocation and accounting scenario, not evidence of a profitable trading strategy. No return, signal quality, or statistical significance can be inferred from the target allocations alone.



### **Analytical Interpretation**

**1. Target-Ledger Separation.** The caller-supplied target mapping is not directly stored as live state; the published state contains immutable position tuples.

**2. Multi-Target Assignment.** The engine uses parallel-style assignment for local scalar calculations where the relationships are easy to review.

**3. Capital Recycling.** Selling 30 AAPL shares produces &#36;2,997 net after fees; this helps fund MSFT and JPM purchases.

**4. Cash Conservation.** Final cash is USD 492.50, and final equity differs from initial equity by the modeled USD 7.50 transaction fees.

**5. Rollback Behavior.** An oversized target causes validation failure before state publication. No partial position or cash update becomes authoritative.

**6. No-Op Semantics.** Repeating the same targets produces no trades and does not increment the ledger version.

**7. Research Engineering.** The result separates input capture, candidate construction, financial reconciliation, and state publication—an important distinction when coding backtests and portfolio state transitions.

**8. Limitations.** The example does not model varying execution prices, cash settlement timing, short selling, margin, market impact, fractional shares, or persistent ledger audit trails.


## 2.3.11 Automated Testing and Financial Invariants

### Test Objectives

The tests assess both Python assignment semantics and the portfolio engine's application-level guarantees.

### Assignment Invariants

1. RHS evaluation in a chained assignment occurs once.
2. All targets of a chained assignment receive the same RHS object.
3. Swapping captures previous values before target updates.
4. Overlapping assignment targets are applied left to right.
5. Repeated nested mutable containers can alias unexpectedly.
6. A later target failure may leave an earlier target modified.

### Portfolio Invariants

1. Nonnegative cash and holdings for committed long-only states.
2. Every held asset has a valid positive mark price during a rebalance.
3. `delta = target_shares - old_shares` for every trade.
4. Fee in USD equals trade notional multiplied by the fee rate in basis points.
5. `pre_equity = post_equity + total_fees`, within stated tolerances.
6. Invalid requests do not publish a new state.
7. No-op requests preserve state and version.
8. Caller-supplied mutable inputs remain unchanged.
9. Immutable snapshots do not change after subsequent updates.
10. Equal input states and targets produce deterministic results under the same price assumptions.

### Test Interpretation

A passing unit-test suite does not establish production readiness. External I/O, crash recovery, concurrent input mutation, accounting-grade precision, and historical market execution realism are outside the scope of the implementation.


In [13]:
import unittest


class MultipleAssignmentAndRebalanceTests(unittest.TestCase):
    """Verify assignment semantics and staged portfolio accounting."""

    def setUp(self) -> None:
        self.prices = {"AAPL": 100.0, "MSFT": 200.0, "JPM": 150.0}
        self.target = {"AAPL": 70, "MSFT": 65, "JPM": 10}
        self.book = RebalanceBook(2_000.0, {"AAPL": 100, "MSFT": 50})

    def test_single_chained_rhs_evaluation(self) -> None:
        calls = []
        def build():
            calls.append("called")
            return []
        first = second = build()
        self.assertIs(first, second)
        self.assertEqual(calls, ["called"])

    def test_multiple_independent_mutable_values(self) -> None:
        first, second = [], []
        self.assertIsNot(first, second)

    def test_swap_and_rotation(self) -> None:
        a, b, c = 1, 2, 3
        a, b, c = b, c, a
        self.assertEqual((a, b, c), (2, 3, 1))

    def test_overlapping_target_order(self) -> None:
        numbers, index = [0, 1], 0
        index, numbers[index] = 1, 99
        self.assertEqual(numbers, [0, 99])

    def test_reference_cash_and_fees(self) -> None:
        report = self.book.rebalance(self.target, self.prices)
        self.assertAlmostEqual(report.after.cash_usd, 492.5)
        self.assertAlmostEqual(report.total_fees_usd, 7.5)

    def test_target_positions(self) -> None:
        report = self.book.rebalance(self.target, self.prices)
        self.assertEqual(dict(report.after.positions), self.target)
        self.assertEqual(report.after.version, 1)

    def test_equity_reconciliation(self) -> None:
        report = self.book.rebalance(self.target, self.prices)
        self.assertTrue(isclose(
            report.pre_equity_usd,
            report.post_equity_usd + report.total_fees_usd,
            rel_tol=1e-12,
            abs_tol=1e-8,
        ))

    def test_missing_price_rejected_without_mutation(self) -> None:
        before = self.book.snapshot()
        with self.assertRaises(ValueError):
            self.book.rebalance(self.target, {"AAPL": 100.0, "MSFT": 200.0})
        self.assertEqual(self.book.snapshot(), before)

    def test_oversized_target_rollback(self) -> None:
        before = self.book.snapshot()
        with self.assertRaises(ValueError):
            self.book.rebalance({"AAPL": 1_000, "MSFT": 50}, self.prices)
        self.assertEqual(self.book.snapshot(), before)

    def test_invalid_boolean_quantity(self) -> None:
        with self.assertRaises(ValueError):
            self.book.rebalance({"AAPL": True}, self.prices)

    def test_nonfinite_price_rejected(self) -> None:
        with self.assertRaises(ValueError):
            self.book.rebalance(self.target, {**self.prices, "AAPL": float("nan")})

    def test_noop_version_unchanged(self) -> None:
        first = self.book.rebalance(self.target, self.prices)
        noop = self.book.rebalance(self.target, self.prices)
        self.assertFalse(noop.committed)
        self.assertIs(noop.after, first.after)
        self.assertEqual(self.book.snapshot().version, 1)

    def test_caller_inputs_are_preserved(self) -> None:
        original_target = dict(self.target)
        original_prices = dict(self.prices)
        self.book.rebalance(self.target, self.prices)
        self.assertEqual(self.target, original_target)
        self.assertEqual(self.prices, original_prices)

    def test_old_snapshot_is_unchanged(self) -> None:
        old = self.book.snapshot()
        self.book.rebalance(self.target, self.prices)
        self.assertEqual(old.version, 0)
        self.assertEqual(old.cash_usd, 2_000.0)
        self.assertEqual(dict(old.positions), {"AAPL": 100, "MSFT": 50})

    def test_sale_funds_purchases(self) -> None:
        report = self.book.rebalance(self.target, self.prices)
        self.assertGreaterEqual(report.after.cash_usd, 0.0)
        self.assertTrue(any(trade.delta_shares < 0 for trade in report.trades))
        self.assertTrue(any(trade.delta_shares > 0 for trade in report.trades))

    def test_full_liquidation(self) -> None:
        liquidate = self.book.rebalance({}, self.prices)
        self.assertEqual(liquidate.after.positions, ())
        self.assertAlmostEqual(liquidate.after.cash_usd, 21_980.0)
        self.assertAlmostEqual(liquidate.total_fees_usd, 20.0)

    def test_distinct_books_do_not_share_positions(self) -> None:
        first = RebalanceBook(2_000.0, {"AAPL": 100, "MSFT": 50})
        second = RebalanceBook(2_000.0, {"AAPL": 100, "MSFT": 50})
        first.rebalance(self.target, self.prices)
        self.assertNotEqual(first.snapshot(), second.snapshot())
        self.assertEqual(second.snapshot().version, 0)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(
    MultipleAssignmentAndRebalanceTests
)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()


test_caller_inputs_are_preserved (__main__.MultipleAssignmentAndRebalanceTests.test_caller_inputs_are_preserved) ... 

ok


test_distinct_books_do_not_share_positions (__main__.MultipleAssignmentAndRebalanceTests.test_distinct_books_do_not_share_positions) ... 

ok


test_equity_reconciliation (__main__.MultipleAssignmentAndRebalanceTests.test_equity_reconciliation) ... 

ok


... (25 outputs left out)

ok


test_target_positions (__main__.MultipleAssignmentAndRebalanceTests.test_target_positions) ... 

ok


----------------------------------------------------------------------
Ran 17 tests in 0.009s

OK


### **Analytical Interpretation**

**1. Assignment Semantics.** Unit tests verify the single RHS evaluation for chained assignment, independence of two constructed mutable values, swapping behavior, and overlapping target order.

**2. Financial Reference Values.** The engine reproduces the expected ending cash (USD 492.50), fees (USD 7.50), and target holdings.

**3. Accounting Identity.** The equity conservation test independently verifies that modeled fees explain the difference between pre-trade and post-trade equity.

**4. Rollback.** Invalid targets, missing prices, and oversized purchases do not publish a new state under expected validation paths.

**5. No-Op.** A repeated identical target does not change the version or create additional fees.

**6. Snapshot Isolation.** A pre-rebalance immutable state remains unchanged when a later state is published.

**7. Multi-Portfolio Independence.** Separate books are constructed with independent states, avoiding the aliasing problems illustrated earlier in the notebook.

**8. Explicit Limitations.** These deterministic tests do not establish resilience to process failure, market data revisions, unsupported external mutation, or distributed transaction execution.


## 2.3.12 Final Analytical Interpretation and Engineering Trade-offs

### 1. Right-Hand-Side Semantics

Python evaluates the right-hand side before applying ordinary assignment targets. In a chained assignment, the expression is evaluated once and its object is assigned to each target.

### 2. Multiple vs. Chained Assignment

`a, b = expression_a, expression_b` constructs a target list that distributes two results. `a = b = expression` assigns one evaluated object to both names.

### 3. Object Identity

Chained assignments can intentionally share mutable or immutable objects. For mutable objects, this may expose unintended shared state; independent construction or explicit copying may be needed.

### 4. Evaluation and Target Order

Right-hand-side evaluation occurs before target assignment, but the resulting target assignments execute from left to right. Earlier target modifications may alter later target expressions.

### 5. Partial Updates

Assignment syntax does not guarantee transaction atomicity. A late target failure may leave earlier target writes intact. Nested unpacking can exhibit similar partial-update behavior.

### 6. Iterators and Unpacking

Failed unpacking can still consume generators or produce side effects. Failure atomicity must not be inferred from unchanged simple name targets.

### 7. Static Code Analysis

AST nodes distinguish top-level chained targets from nested target-list syntax. Static analysis can identify suspicious patterns but cannot prove the runtime identities of referenced objects.

### 8. Quantitative Research Reproducibility

Shared mutable objects can contaminate backtesting, scenario analysis, and audit records. Independent data construction and immutable snapshots are important safeguards.

### 9. Financial State Transitions

The rebalancing engine demonstrates the distinction between local paired assignments and publishing an immutable portfolio state. Candidate-state validation and equity conservation help preserve consistency.

### 10. Synchronization, Atomicity, and Durability

A lock protecting a single in-process state reference is not equivalent to durable transactions or distributed consistency. These properties require additional systems engineering.

### 11. Numerical Accuracy

Binary floating-point is used for this educational model. Production monetary accounting may need decimal precision, explicit currency conventions, rounding rules, and reconciliation with authoritative ledgers.

### 12. Practical Design Rule

Use multiple assignment for concise, coherent transformations. Use chained assignment only when sharing an object is intentional. Use staged immutable state and explicit invariants for coordinated financial updates.

### Final Assessment

Advanced assignment expertise requires understanding not just the syntax but the objects being shared, the exact order of evaluation and target updates, the failure boundaries, and the consistency guarantees imposed by the surrounding application.


## Advanced Practice Exercises — Expert Level

**Instructions:** Complete each exercise in a separate Code Cell or an independent practice notebook. Provide an automated test suite and a short `### **Analytical Interpretation**` explaining the outcome, failure conditions, and financial/software-engineering relevance.

### Exercise 1 — Assignment Semantics Trace Analyzer

**Objective:** Build an execution tracer that records all RHS calls, target-expression evaluations, and setter invocations for mixed target lists.

**Tasks:**
1. Trace a two-target tuple assignment and a three-target chained assignment.
2. Verify the RHS evaluation count.
3. Include two subscription targets and one property setter.
4. Demonstrate one assignment that fails at the final target.
5. Show which earlier state modifications remain visible.
6. Compare observations with the language-level evaluation rules.

**Requirements:** standard library only; deterministic assertions; explicit side-effect logs.

### Exercise 2 — Mutable Alias Graph Detector

**Objective:** Identify selected shared nested mutable objects in research containers.

**Tasks:**
1. Construct deliberately shared dictionaries and lists.
2. Detect aliases using object identity without relying on persistent `id()` values.
3. Detect sharing introduced by chained assignment and list multiplication.
4. Compare against separately constructed containers.
5. Generate an object-sharing report.
6. Discuss recursion, cycles, and false positives.

**Requirements:** handle nested dict/list structures; no claim of complete heap introspection.

### Exercise 3 — Overlapping Targets and State Mutation

**Objective:** Predict the effects of complicated left-hand target ordering.

**Tasks:**
1. Construct five overlapping-target examples.
2. Predict results before execution.
3. Record target-expression evaluation order.
4. Include mixed name, attribute, and item targets.
5. Introduce a setter that raises an exception.
6. Inspect final state after the exception.
7. Refactor each expression for maximum clarity.

**Requirements:** compare predicted and actual results with tests.

### Exercise 4 — Unpacking Failure Boundary Investigation

**Objective:** Differentiate extraction failures from target-assignment failures.

**Tasks:**
1. Test too few and too many RHS values.
2. Build nested target patterns that partially update outer names.
3. Implement generators with visible side effects.
4. Analyze where exceptions are raised.
5. Document which bindings or objects have changed.
6. Propose a validated-input alternative.

**Requirements:** deterministic tests; avoid executing untrusted source.

### Exercise 5 — AST-Based Assignment Code Auditor

**Objective:** Build a static auditor using Python's `ast` module.

**Tasks:**
1. Distinguish chained assignments from tuple-target assignments.
2. Find assignments where the same mutable literal is shared across targets.
3. Identify selected overlapping name/subscript targets.
4. Flag source lines that appear to change several portfolio state fields.
5. Record source locations and node types.
6. Produce review recommendations and describe detection limitations.

**Requirements:** parse without executing; include nested function definitions.

### Exercise 6 — Portfolio Rebalance Sensitivity Analysis

**Objective:** Extend the portfolio engine to compare several target portfolios and fee scenarios.

**Tasks:**
1. Generate independent target allocations.
2. Evaluate fees from 0 to 50 basis points.
3. Rank target requests by execution costs and cash feasibility.
4. Verify equity conservation for every accepted scenario.
5. Ensure rejected scenarios preserve the starting snapshot.
6. Record deterministic result tables.
7. Explain why cost minimization does not imply investment optimality.

**Requirements:** clear units; no hidden mutable references between scenarios.

### Exercise 7 — Transactional Publication and Concurrency

**Objective:** Compare multi-attribute updates against single immutable-state publication.

**Tasks:**
1. Build one intentionally unsafe state object with separate cash and position attributes.
2. Construct a deterministic failure after the cash update.
3. Observe inconsistent state.
4. Refactor to staged immutable candidates and one published reference.
5. Add synchronization for all cooperating readers and writers.
6. Test concurrent rebalance requests with controlled test inputs.
7. Document the limits of the GIL, locks, and in-memory atomicity.

**Requirements:** no claims of process-crash or cross-process durability.

### Exercise 8 — Reproducible Quant Research Inputs

**Objective:** Prevent multi-assignment aliasing errors in financial experiment preparation.

**Tasks:**
1. Create independent in-sample and out-of-sample parameter dictionaries.
2. Demonstrate a contaminating chained-assignment anti-pattern.
3. Correct it using independent data construction.
4. Construct immutable experiment snapshots.
5. Compare experiment outputs under deliberate source mutation.
6. Verify unchanged snapshots and explain data versioning.

**Requirements:** deterministic data and calculations; documented scope of immutability.


## Advanced Challenge — Assignment-Safe Multi-Portfolio Research Infrastructure

### Objective

Develop a modular quantitative research infrastructure that combines assignment semantics, state isolation, multi-asset rebalancing, versioned snapshots, and deterministic accounting.

### Required Components

**1. Assignment Auditor**
- Distinguish ordinary, chained, tuple-target, nested, and starred assignments.
- Identify selected aliases and overlapping targets.
- Produce source-level audit findings and code-review guidance.

**2. Immutable Research Inputs**
- Validate portfolio holdings and target allocations.
- Construct independent, versioned portfolio snapshots.
- Reject malformed numeric inputs and ambiguous units.

**3. Portfolio Rebalance Planner**
- Calculate signed trade deltas and notional values.
- Model user-configurable transaction fees.
- Support multiple target portfolios and deterministic scenario comparisons.
- Explain how sell-first staging affects funding assumptions.

**4. State Publication Engine**
- Stage all calculations before commit.
- Publish validated immutable states under an explicit synchronization policy.
- Reject invalid targets without changing the authoritative portfolio.
- Return coherent snapshots to cooperating readers.

**5. Accounting and Diagnostics**
- Verify pre/post marked equity reconciliation.
- Track cash balances, gross trade notional, and costs.
- Report all failed and no-op requests.
- Add source data and configuration fingerprints.

**6. Automated Quality Assurance**
- Unit and property-oriented tests for assignment order and aliasing.
- Failure-injection tests at multiple operation boundaries.
- Deterministic numerical regression tests.
- Concurrency tests for cooperating operations.
- Ruff formatting/linting and static type checks as project extensions.

### Expected Deliverables

1. Documented modular Python package with public APIs and data contracts.
2. Assignment analysis utilities and a source-audit report.
3. Immutable portfolio snapshots and independent scenario inputs.
4. Rebalance planner and synchronized in-memory state engine.
5. Financial reconciliation and structured result reporting.
6. Automated failure-path, numerical, and concurrency test suites.
7. A GitHub-ready README including example outputs, limitations, and reproduction commands.

### Evaluation Rubric

| Dimension | Weight |
|---|---:|
| Language semantics and aliasing correctness | 25% |
| Financial accounting and numerical validation | 25% |
| State isolation and failure behavior | 20% |
| Testing and reproducibility | 15% |
| Architecture and maintainability | 10% |
| Documentation and auditability | 5% |

### Final Goal

Demonstrate the ability to use Python's concise multiple assignment forms without confusing them with object independence, transaction atomicity, or durable financial state management.


## Notebook Summary and Research Checklist

| Section | Topic | Main outcome |
|---|---|---|
| 2.3.1 | Assignment Forms | Distinguish target lists from chains |
| 2.3.2 | Single RHS Evaluation | One object, left-to-right targets |
| 2.3.3 | Swapping and Rotation | RHS captured before target updates |
| 2.3.4 | Mutable Aliasing | Independent objects vs. shared containers |
| 2.3.5 | Overlap and Failure | Sequential target changes can survive errors |
| 2.3.6 | Evaluation Tracing | Verify observable side-effect ordering |
| 2.3.7 | Unpacking Failure | Nested target updates and iterator effects |
| 2.3.8 | AST & Bytecode | Structural vs. implementation-specific analysis |
| 2.3.9 | Common Pitfalls | Code review and research data integrity |
| 2.3.10 | Practical Application | Immutable portfolio rebalance publication |
| 2.3.11 | Tests & Invariants | Accounting and language behavior verification |
| 2.3.12 | Final Interpretation | Scope and engineering trade-offs |

### Completion Criteria

- [ ] All executable Code Cells run in a fresh kernel, from top to bottom.
- [ ] All assignment semantics are explained without relying on CPython-only identity optimizations.
- [ ] Every significant code example has a `### **Analytical Interpretation**` section.
- [ ] The financial example reconciles initial and final marked equity after fees.
- [ ] Failed and no-op portfolio transitions behave according to their contracts.
- [ ] At least three Advanced Practice Exercises are implemented independently.
- [ ] The notebook is committed to GitHub with a descriptive commit message.

**Next notebook:** `2.4_Sequence_Unpacking_and_Extended_Assignment.ipynb` — a detailed treatment of nested unpacking, starred targets, iterator consumption, assignment patterns, and edge cases.
